# Family Activity Deep Agent — Shared Calendar & Reminder Drafts

A hands-on tutorial of the **Deep Agents** framework with a standalone **MCP tool server** and durable **SQLite** calendar state.

We build a family coordinator that creates, reads, updates, deletes, and restores activities; detects child and parent conflicts; and stores reviewable reminder drafts.

> **Draft-only messaging.** The MVP stores reminder text but does not send SMS messages. Every calendar or reminder mutation pauses for parent approval.

## Architecture at a glance

```
Parent request → Family Coordinator (plans, routes, verifies)
                         │
          ┌──────────────┼──────────────┐
          ▼              ▼              ▼
       Intake         Calendar       Conflict        Reminder
       Agent           Agent          Agent           Agent
          └──────────────┴──────┬───────┴──────────────┘
                                ▼
                      Family Activity MCP Server
                                ▼
                  SQLite: events | reminders | audit_logs
```

| Deep Agents concept | Implementation |
|---|---|
| Planning and routing | Coordinator fast path plus specialist delegation |
| Subagents | Intake, calendar, conflict, and reminder specialists |
| Shared temporary state | `/work`, `/reviews`, and `/final` JSON artifacts |
| Skills | Calendar policy, reminder policy, and family preferences |
| Durable state | SQLite only; each CLI/notebook request is independent |
| Human-in-the-loop | LangGraph interrupts on all write tools |
| Deterministic safety | Repository validates future times, versions, and overlaps |
| Observability | Optional LangSmith trace tree |


![Family Activity Deep Agent architecture](./images/01_architecture.svg)


## 1. Install and API keys

Run this notebook from the repository root with the project virtual environment. Dependencies are managed by `pyproject.toml`. Never print API keys in notebook output.

In [ ]:
# If needed, install the project into the active notebook kernel:
# %pip install -e '.[dev]'


In [ ]:
import os
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv

load_dotenv()
if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError("Add GROQ_API_KEY to .env before running the live agent cells.")

if os.getenv("LANGSMITH_API_KEY", "").strip():
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "family-activity-agent-mvp")
    print("LangSmith tracing: ENABLED")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: disabled")

print("Model:", os.getenv("FAMILY_ACTIVITY_MODEL", "openai/gpt-oss-20b"))


## 2. MCP tools

The Deep Agent launches the local MCP server as a managed stdio subprocess. Tool definitions live in `src/family_activity_mcp/server.py`; validation and SQLite transactions live in `repository.py`.

![MCP tool boundary](./images/02_mcp_tools.svg)


In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from family_activity_agent.agent import mcp_connection

async def discover_tools():
    client = MultiServerMCPClient(mcp_connection())
    return await client.get_tools()

tools = asyncio.run(discover_tools())
for tool in tools:
    print(f"{tool.name}: {tool.description}")


## 3. Skills (progressive disclosure)

The coordinator sees skill descriptions first and loads full `SKILL.md` instructions only when relevant. This keeps the always-on prompt smaller while preserving domain policy.

![Skills progressive disclosure](./images/03_skills.svg)


In [ ]:
for path in sorted(Path("skills").glob("*/SKILL.md")):
    print(f"\n===== {path} =====")
    print(path.read_text())


## 4. State boundary

SQLite is the sole durable source of truth. The checkpointer is in-memory because it is needed only to pause and resume approval within the current process. Temporary planning artifacts are cleared at the start of each request so stale files cannot influence a new command.

![State and memory boundary](./images/04_state.svg)


In [ ]:
from family_activity_agent.cli import RUN_ARTIFACTS

print("Durable state: data/family_activity.db")
print("Ephemeral per-run artifacts:")
for artifact in RUN_ARTIFACTS:
    print(" -", artifact)


## 5. Specialist subagents

Each specialist has a focused prompt and restricted tool set. The coordinator handles simple requests directly to reduce latency and delegates only when specialization adds value.

![Coordinator and specialist agents](./images/05_subagents.svg)


In [ ]:
from family_activity_agent.prompts import (
    INTAKE_PROMPT, CALENDAR_PROMPT, CONFLICT_PROMPT, REMINDER_PROMPT
)

subagents = {
    "intake-agent": "Normalizes ambiguous or multi-activity requests",
    "calendar-agent": "Handles complex event lifecycle operations",
    "conflict-agent": "Explains overlaps without mutating state",
    "reminder-agent": "Creates and manages reminder drafts",
}
for name, purpose in subagents.items():
    print(f"{name}: {purpose}")


## 6. Assemble the Deep Agent

`build_family_agent()` discovers MCP tools, wraps their results for Groq compatibility, registers subagents and skills, and configures interrupts on every mutating tool.

In [ ]:
from family_activity_agent.agent import build_family_agent

agent = asyncio.run(build_family_agent())
print("Deep Agent assembled:", agent.name)


## 7. Run a read-only request

Read tools do not require approval. Use synthetic demo data when LangSmith tracing is enabled.

In [ ]:
read_config = {
    "configurable": {"thread_id": f"notebook-read-{uuid4()}"},
    "recursion_limit": 30,
    "run_name": "family-activity-notebook-read",
    "tags": ["family-activity-agent", "notebook", "demo"],
}

read_result = asyncio.run(agent.ainvoke(
    {"messages": [{"role": "user", "content": "Show today's activities for family-1"}]},
    config=read_config,
))
read_result["messages"][-1].pretty_print()


## 8. Human-in-the-loop event creation

The next cell proposes a future event. The graph must stop before `create_event`; inspect the exact tool arguments before approving.

![Human-in-the-loop write gate](./images/06_hitl.svg)


In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

demo_day = (datetime.now(ZoneInfo("America/Los_Angeles")) + timedelta(days=7)).date()
write_config = {
    "configurable": {"thread_id": f"notebook-write-{uuid4()}"},
    "recursion_limit": 30,
    "run_name": "family-activity-notebook-write",
    "tags": ["family-activity-agent", "notebook", "demo"],
}
prompt = (
    f"Add Demo Child's soccer practice on {demo_day.isoformat()} "
    "from 4 to 5 PM for family-1"
)
write_result = asyncio.run(agent.ainvoke(
    {"messages": [{"role": "user", "content": prompt}]},
    config=write_config,
))
print("Paused for approval:", "__interrupt__" in write_result)


In [ ]:
from family_activity_agent.cli import pending_requests

requests = pending_requests(write_result)
for request in requests:
    print("PENDING APPROVAL:", request)


### Approve or reject explicitly

Run the approval cell only after reviewing the request above. To reject, replace the decision with `{"type": "reject", "message": "Reason"}`.

In [ ]:
from langgraph.types import Command

if not requests:
    print("No mutation is waiting for approval.")
else:
    decisions = [{"type": "approve"} for _ in requests]
    write_result = asyncio.run(agent.ainvoke(
        Command(resume={"decisions": decisions}),
        config=write_config,
    ))
    write_result["messages"][-1].pretty_print()


## 9. Verify durable state

Use a separate read request. The new thread has no conversation history, so seeing the event proves that SQLite—not memory—is carrying the state.

In [ ]:
verify_config = {
    "configurable": {"thread_id": f"notebook-verify-{uuid4()}"},
    "recursion_limit": 30,
}
verify_result = asyncio.run(agent.ainvoke(
    {"messages": [{"role": "user", "content": f"Show activities on {demo_day.isoformat()} for family-1"}]},
    config=verify_config,
))
verify_result["messages"][-1].pretty_print()


## 10. Conflict safety

Different children may have overlapping activities. Assigning overlapping events to the same parent is rejected deterministically by the repository, even if the model skips an explanatory check.

![Deterministic conflict safety](./images/07_conflicts.svg)


In [ ]:
print(
    "Try in the CLI:",
    'family-activity-agent "Assign two overlapping activities in family-1 to parent-1"',
    sep="\n",
)


## 11. Reminder drafts

The reminder tool stores recipient IDs, scheduled time, channel, and `message_body`. It does **not** call Twilio and the agent must never promise delivery.

In [ ]:
print(
    "Example request:",
    "Draft a day-of reminder at 8 AM Pacific for both parents "
    "for Demo Child's next soccer practice in family-1",
    sep="\n",
)


## 12. Offline evaluations

The automated suite uses a deterministic fake chat model but retains the actual Deep Agent, LangGraph interrupts, MCP subprocess, and SQLite layers. This avoids Groq quota use in CI.

![Offline end-to-end evaluations](./images/08_evaluations.svg)


In [ ]:
# Run from a notebook cell if desired:
# !pytest -q

import json
cases = json.loads(Path("evaluations/cases.json").read_text())
print(f"Evaluation cases: {len(cases)}")
for case in cases[:5]:
    print(f"- {case['id']}: {case['expected_outcome']}")


## 13. LangSmith observability

When enabled, the trace shows the coordinator at the root, model calls and MCP tools as nested spans, and approval/resumption across the workflow. Use only synthetic family data in traced demonstrations.

![LangSmith trace tree](./images/09_langsmith.svg)


In [ ]:
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT", "not configured"))
print("Tracing enabled:", os.getenv("LANGSMITH_TRACING", "false"))


## Recap

You built a family activity system that demonstrates the course's core agentic requirements:

- A **Deep Agent coordinator** that routes and delegates.
- Four **specialist subagents** with focused prompts and tools.
- Ten typed **MCP tools** backed by SQLite.
- **Human approval** before every write.
- **Deterministic safety** for past times, conflicts, versions, and idempotency.
- **Draft-only reminders** with no external messaging side effects.
- A deliberate **state boundary**: durable database, ephemeral conversation.
- **Offline evaluations** and optional **LangSmith tracing**.

**Where to take it next:** add a shared calendar frontend, authenticated family membership, Google Calendar synchronization, and an optional Twilio worker after the draft-only MVP is accepted.